# DeepMPP Automation — Optical Property Prediction

Predict molecular absorption (Abs) and emission (Emi) peaks using a pre-trained GATwithSolv model.

Input mode:
- **Batch mode**: specify a folder containing CSV files with the same format; all matched files will be processed automatically.

Input CSV requirement: must contain a `SMILES` column.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm

# ============================================================
# User Configuration (modify this section as needed)
# ============================================================

# Input folder path (contains CSV files with the same format; each CSV must have a SMILES column)
INPUT_DIR = "/home/cenking/VsCode/Test/TSte"

# Input file matching pattern
INPUT_PATTERN = "BatchE003Slice*.csv"

# Output folder path
OUTPUT_DIR = "/home/cenking/VsCode/zye"

# Default solvent SMILES (DMSO)
DEFAULT_SOLVENT = "CS(C)=O"

# DeepMPP project path
DEEPMPP_DIR = "/home/cenking/VsCode/DeepMPP"
sys.path.insert(0, DEEPMPP_DIR)

# GAT model path (absolute path)
MODEL_PATH = "/home/cenking/VsCode/DeepMPP/models/GAT_models/_Models/GATwithSolv_model_optical_train_Abs,Emi_20260415_175307"

# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input:   {INPUT_DIR}")
print(f"Pattern: {INPUT_PATTERN}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Solvent: {DEFAULT_SOLVENT} (DMSO)")
print(f"Model:   {MODEL_PATH}")

## 1. Load Model

In [ ]:
import os
import yaml

os.chdir(os.path.join(DEEPMPP_DIR, "models", "GAT_models", "_Models"))

# ---- Force model config device to cpu ----
model_folder = "GATwithSolv_model_optical_train_Abs,Emi_20260415_175307"
config_path = os.path.join(model_folder, "config.yaml")

with open(config_path, "r") as f:
    config_data = yaml.safe_load(f)

original_device = config_data.get("device", "cpu")
if original_device != "cpu":
    config_data["device"] = "cpu"
    with open(config_path, "w") as f:
        yaml.dump(config_data, f, default_flow_style=False)
    print(f"Changed config.yaml device from '{original_device}' to 'cpu'")

# ---- Load model ----
from D4CMPP import Analyzer

ma = Analyzer.MolAnalyzer(model_folder)
print("Model loaded successfully")
print(f"Target properties: {ma.config['target']}")
print(f"Device: {ma.config['device']}")

## 2. Define Prediction Functions

In [ ]:
def predict_file(analyzer, csv_path, solvent_smiles=DEFAULT_SOLVENT, batch_size=512):
    """Predict a single CSV file and return a DataFrame with prediction results."""
    df = pd.read_csv(csv_path)
    smiles_list = df["SMILES"].tolist()
    solvent_list = [solvent_smiles] * len(smiles_list)

    all_abs, all_emi = [], []
    failed_idx = []

    for i in range(0, len(smiles_list), batch_size):
        batch_smiles = smiles_list[i:i+batch_size]
        batch_solvent = solvent_list[i:i+batch_size]
        try:
            result = analyzer.predict(batch_smiles, batch_solvent)
            for j, smi in enumerate(batch_smiles):
                key = (smi, solvent_smiles)
                if key in result:
                    pred = result[key]
                    all_abs.append(pred[0])
                    all_emi.append(pred[1])
                else:
                    all_abs.append(np.nan)
                    all_emi.append(np.nan)
                    failed_idx.append(i + j)
        except Exception as e:
            print(f"  Batch {i}-{i+len(batch_smiles)} error: {e}")
            for j in range(len(batch_smiles)):
                all_abs.append(np.nan)
                all_emi.append(np.nan)
                failed_idx.append(i + j)

    df["Abs_pred"] = all_abs
    df["Emi_pred"] = all_emi
    df["Solvent"] = solvent_smiles

    if failed_idx:
        print(f"  Warning: {len(failed_idx)} molecules failed")

    return df


def predict_folder(analyzer, input_dir, output_dir, solvent_smiles=DEFAULT_SOLVENT, pattern="*.csv", batch_size=512):
    """Batch-process all matching CSV files in a folder; results are saved separately."""
    csv_files = sorted(glob(os.path.join(input_dir, pattern)))

    if len(csv_files) == 0:
        print(f"No files matching '{pattern}' found in {input_dir}")
        return

    print(f"{len(csv_files)} file(s) to predict")
    print(f"Input:   {input_dir}")
    print(f"Output:  {output_dir}")
    print(f"Solvent: {solvent_smiles}\n")

    total_molecules = 0
    total_valid = 0
    total_failed = 0

    for csv_path in tqdm(csv_files, desc="Prediction progress"):
        filename = os.path.basename(csv_path)
        df_pred = predict_file(analyzer, csv_path, solvent_smiles, batch_size)

        out_path = os.path.join(output_dir, filename.replace(".csv", "_pred.csv"))
        df_pred.to_csv(out_path, index=False)

        total_molecules += len(df_pred)
        total_valid += df_pred["Abs_pred"].notna().sum()
        total_failed += df_pred["Abs_pred"].isna().sum()

    print(f"\nPrediction complete! Total: {total_molecules} molecules")
    print(f"Valid predictions: {total_valid}")
    print(f"Failed predictions: {total_failed}")

## 3. Batch Prediction

In [ ]:
predict_folder(
    analyzer=ma,
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    solvent_smiles=DEFAULT_SOLVENT,
    pattern=INPUT_PATTERN,
    batch_size=512
)